# Cross-Architecture Analysis

**Hypotheses tested:**
- **H2**: Architecture family matters more than model size for image-heavy tasks (Matrix Reasoning, TROG, Vocab) because the vision encoder is the bottleneck.
- **H4**: Accuracy on text-only tasks (EGMA Math, Theory of Mind) scales more smoothly with model size than accuracy on image-option tasks.

**Models compared:**

| Model | Family | Params | Vision encoder |
|-------|--------|--------|----------------|
| SmolVLM2-500M | smolvlm2 | 0.5B | SigLIP-400M |
| TinyLLaVA-3.1B | tinyllava | 3.1B | SigLIP-400M |
| Qwen3.5-0.8B | qwen35 | 0.8B | ViT |
| Qwen3.5-4B | qwen35 | 4.0B | ViT |
| InternVL3.5-2B | internvl35 | 2.0B | ViT-300M |
| InternVL3.5-8B | internvl35 | 8.0B | ViT-300M |

**Tasks:**

| Task | Option type | Difficulty driver |
|------|-------------|-------------------|
| egma-math | text (numbers) | Arithmetic reasoning |
| theory-of-mind | text (phrases) | Narrative reasoning |
| vocab | images | Visual vocabulary |
| trog | images | Grammar + scene understanding |
| matrix-reasoning | images + context | Visual pattern completion |

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').exists() else _cwd.parent
RESULTS_DIR = ROOT / 'results' / 'cross_architecture'

# Model metadata — order determines plot order
MODEL_META = [
    dict(name='smolvlm2_500m',  family='smolvlm2',   params_b=0.5,  label='SmolVLM2\n0.5B',  encoder='SigLIP'),
    dict(name='qwen35_0.8b',    family='qwen35',     params_b=0.8,  label='Qwen3.5\n0.8B',   encoder='ViT'),
    dict(name='internvl35_2b',  family='internvl35', params_b=2.0,  label='InternVL3.5\n2B', encoder='ViT-300M'),
    dict(name='tinyllava_3.1b', family='tinyllava',  params_b=3.1,  label='TinyLLaVA\n3.1B', encoder='SigLIP'),
    dict(name='qwen35_4b',      family='qwen35',     params_b=4.0,  label='Qwen3.5\n4B',     encoder='ViT'),
    dict(name='internvl35_8b',  family='internvl35', params_b=8.0,  label='InternVL3.5\n8B', encoder='ViT-300M'),
]
meta_df = pd.DataFrame(MODEL_META).set_index('name')

TASK_ORDER  = ['egma-math', 'theory-of-mind', 'vocab', 'trog', 'matrix-reasoning']
TASK_LABELS = ['EGMA Math', 'Theory of Mind', 'Vocab', 'TROG', 'Matrix Reasoning']
TASK_TYPE   = ['text', 'text', 'image', 'image', 'image']   # option modality

FAMILY_COLORS = {
    'smolvlm2':   '#4C72B0',
    'qwen35':     '#DD8452',
    'internvl35': '#55A868',
    'tinyllava':  '#C44E52',
}

print(f'Results root: {RESULTS_DIR}')
print(f'Exists: {RESULTS_DIR.exists()}')

In [ ]:
def load_results(results_dir: Path, model_names: list, task_ids: list) -> pd.DataFrame:
    """Load per-trial CSVs for all models and tasks into a single DataFrame."""
    rows = []
    for model in model_names:
        # Find version directory (only one expected)
        model_dirs = list((results_dir / model).glob('*')) if (results_dir / model).exists() else []
        if not model_dirs:
            print(f'  MISSING results for {model}')
            continue
        version_dir = sorted(model_dirs)[-1]  # most recent
        for task in task_ids:
            task_csv = version_dir / f'{task}.csv'
            if not task_csv.exists():
                print(f'  MISSING {model}/{task}')
                continue
            df = pd.read_csv(task_csv)
            df['model'] = model
            df['task']  = task
            rows.append(df)
    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)

model_names = [m['name'] for m in MODEL_META]
results = load_results(RESULTS_DIR, model_names, TASK_ORDER)

if results.empty:
    print('\nNo results found yet — run scripts/run_cross_architecture.py first.')
else:
    print(f'Loaded {len(results):,} trial results')
    print(results.groupby(['model','task'])['is_correct'].count().unstack())

## 1. Accuracy heatmap — model × task

Primary view: which model × task combinations perform well. Models are sorted by parameter count. Tasks are grouped by option modality (text vs image).

In [ ]:
if not results.empty:
    acc = (results.groupby(['model', 'task'])['is_correct']
                  .mean()
                  .unstack('task')
                  .reindex(index=model_names, columns=TASK_ORDER))

    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(acc.values, vmin=0, vmax=1, cmap='RdYlGn', aspect='auto')

    ax.set_xticks(range(len(TASK_ORDER)))
    ax.set_xticklabels(TASK_LABELS, fontsize=11)
    ax.set_yticks(range(len(model_names)))
    ax.set_yticklabels([meta_df.loc[m, 'label'] for m in model_names], fontsize=10)

    # Annotate cells with accuracy values
    for i, model in enumerate(model_names):
        for j, task in enumerate(TASK_ORDER):
            val = acc.loc[model, task]
            if pd.notna(val):
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        fontsize=10, color='black' if 0.35 < val < 0.75 else 'white',
                        fontweight='bold')

    # Separator between text and image tasks
    ax.axvline(1.5, color='white', linewidth=2)
    ax.text(0.5, -0.7, '← text options', ha='center', transform=ax.get_xaxis_transform(),
            fontsize=9, color='gray')
    ax.text(3.0, -0.7, 'image options →', ha='center', transform=ax.get_xaxis_transform(),
            fontsize=9, color='gray')

    plt.colorbar(im, ax=ax, label='Accuracy')
    ax.set_title('Accuracy by model and task', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(ROOT / 'results' / 'cross_architecture' / 'heatmap_accuracy.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print(acc.to_string(float_format='{:.3f}'.format))

## 2. Scaling curves — accuracy vs parameters by task type

**H4 test**: Do text-option tasks scale more smoothly with model size than image-option tasks?

- Each point = one model
- Color = architecture family
- Left panel = text tasks | Right panel = image tasks

In [ ]:
if not results.empty:
    acc_df = (results.groupby(['model', 'task'])['is_correct']
                     .mean()
                     .reset_index(name='accuracy')
                     .merge(meta_df.reset_index(), on='model'))

    text_tasks  = [t for t, typ in zip(TASK_ORDER, TASK_TYPE) if typ == 'text']
    image_tasks = [t for t, typ in zip(TASK_ORDER, TASK_TYPE) if typ == 'image']
    task_label  = dict(zip(TASK_ORDER, TASK_LABELS))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

    for ax, task_group, title in zip(
        axes,
        [text_tasks, image_tasks],
        ['Text-option tasks', 'Image-option tasks']
    ):
        for task in task_group:
            sub = acc_df[acc_df['task'] == task].sort_values('params_b')
            ax.plot(sub['params_b'], sub['accuracy'], 'o--', alpha=0.5,
                    label=f'_{task}', color='lightgray', linewidth=1)

        for _, row in acc_df[acc_df['task'].isin(task_group)].iterrows():
            ax.scatter(row['params_b'], row['accuracy'],
                       color=FAMILY_COLORS[row['family']],
                       s=120, zorder=5,
                       label=row['family'] if row['family'] not in
                       [l.get_label() for l in ax.get_lines()] else '')

        ax.axhline(0.25, color='gray', linestyle=':', linewidth=1, label='chance (25%)')
        ax.set_xlabel('Parameters (B)', fontsize=11)
        ax.set_ylabel('Accuracy', fontsize=11)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_ylim(0, 1.05)
        ax.set_xscale('log')

    # Family legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=f) for f, c in FAMILY_COLORS.items()]
    fig.legend(handles=legend_elements, loc='lower center', ncol=4,
               bbox_to_anchor=(0.5, -0.05), fontsize=10, title='Architecture family')

    plt.tight_layout()
    plt.savefig(ROOT / 'results' / 'cross_architecture' / 'scaling_curves.png',
                dpi=150, bbox_inches='tight')
    plt.show()

## 3. Architecture family vs size — H2 test

Compare average accuracy on **image tasks** for models within the same family (size effect) vs models of similar size from different families (architecture effect).

- **Family effect**: InternVL3.5-2B vs InternVL3.5-8B (same encoder, different LLM)
- **Architecture effect at ~2–3B**: InternVL3.5-2B vs TinyLLaVA-3.1B vs Qwen3.5-4B (similar size, different encoders)

In [ ]:
if not results.empty:
    image_results = results[results['task'].isin(image_tasks)]
    img_acc = (image_results.groupby('model')['is_correct']
                             .mean()
                             .reset_index(name='img_accuracy')
                             .merge(meta_df.reset_index(), on='model')
                             .sort_values('params_b'))

    text_results = results[results['task'].isin(text_tasks)]
    txt_acc = (text_results.groupby('model')['is_correct']
                            .mean()
                            .reset_index(name='txt_accuracy')
                            .merge(meta_df.reset_index(), on='model'))

    combined = img_acc.merge(txt_acc[['name','txt_accuracy']], on='name')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, col, title in zip(
        axes,
        ['img_accuracy', 'txt_accuracy'],
        ['Avg accuracy — image tasks', 'Avg accuracy — text tasks']
    ):
        bars = ax.bar(
            range(len(combined)),
            combined[col],
            color=[FAMILY_COLORS[f] for f in combined['family']],
            edgecolor='white', linewidth=0.8
        )
        ax.set_xticks(range(len(combined)))
        ax.set_xticklabels(
            [f"{row['label']}\n({row['encoder']})" for _, row in combined.iterrows()],
            fontsize=9
        )
        ax.axhline(0.25, color='gray', linestyle=':', linewidth=1)
        ax.set_ylim(0, 1.0)
        ax.set_ylabel('Accuracy', fontsize=11)
        ax.set_title(title, fontsize=12, fontweight='bold')
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

    legend_elements = [Patch(facecolor=c, label=f) for f, c in FAMILY_COLORS.items()]
    fig.legend(handles=legend_elements, loc='lower center', ncol=4,
               bbox_to_anchor=(0.5, -0.06), fontsize=10, title='Architecture family')
    plt.tight_layout()
    plt.savefig(ROOT / 'results' / 'cross_architecture' / 'family_vs_size.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # Summary table
    print(combined[['label','family','params_b','encoder','img_accuracy','txt_accuracy']]
          .rename(columns={'img_accuracy':'img_acc','txt_accuracy':'txt_acc'})
          .to_string(index=False, float_format='{:.3f}'.format))

## 4. Human plurality alignment

`human_plurality_agrees_model`: fraction of trials where the model's predicted label matches the most common human choice. A model that aligns with the human plurality makes the same *systematic errors* as children — a richer signal than accuracy alone.

In [ ]:
if not results.empty and 'human_plurality_agrees_model' in results.columns:
    align = (results
             .dropna(subset=['human_plurality_agrees_model'])
             .groupby(['model', 'task'])['human_plurality_agrees_model']
             .mean()
             .unstack('task')
             .reindex(index=model_names, columns=TASK_ORDER))

    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(align.values, vmin=0, vmax=1, cmap='YlOrRd', aspect='auto')

    ax.set_xticks(range(len(TASK_ORDER)))
    ax.set_xticklabels(TASK_LABELS, fontsize=11)
    ax.set_yticks(range(len(model_names)))
    ax.set_yticklabels([meta_df.loc[m, 'label'] for m in model_names if m in align.index], fontsize=10)

    for i, model in enumerate(model_names):
        if model not in align.index:
            continue
        for j, task in enumerate(TASK_ORDER):
            val = align.loc[model, task] if task in align.columns else float('nan')
            if pd.notna(val):
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        fontsize=10, color='black' if val < 0.7 else 'white', fontweight='bold')

    ax.axvline(1.5, color='white', linewidth=2)
    plt.colorbar(im, ax=ax, label='Human plurality agreement rate')
    ax.set_title('Human plurality agreement — model × task', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(ROOT / 'results' / 'cross_architecture' / 'heatmap_human_alignment.png',
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Human alignment columns not present in results (run with human data enabled).')

## 5. Developmental profile — estimated child age equivalent

Uses the IRT calibration from the developmental alignment experiment to map each model's accuracy on Vocab to an estimated child age. Requires the IRT calibration data.

> **Note:** IRT-based age estimation is only valid for vocab in the current setup. Matrix-reasoning and TROG calibration requires running the IRT models on the children's trial data for those tasks (future work).

In [ ]:
if not results.empty:
    vocab_acc = (results[results['task'] == 'vocab']
                 .groupby('model')['is_correct']
                 .mean()
                 .reset_index(name='vocab_acc')
                 .merge(meta_df.reset_index(), on='model')
                 .sort_values('params_b'))

    # Simple linear mapping from vocab accuracy to approximate age (months)
    # Placeholder — replace with actual IRT calibration coefficients from
    # notebooks/vocab_development_alignment.ipynb when available
    # Observed range: chance ~0.25 ≈ 36 months; near-perfect ~1.0 ≈ 96 months
    def acc_to_age_months(acc, chance=0.25, max_age=96, min_age=36):
        corrected = max(0.0, (acc - chance) / (1.0 - chance))
        return min_age + corrected * (max_age - min_age)

    vocab_acc['est_age_months'] = vocab_acc['vocab_acc'].apply(acc_to_age_months)
    vocab_acc['est_age_years']  = vocab_acc['est_age_months'] / 12

    fig, ax = plt.subplots(figsize=(9, 5))
    for _, row in vocab_acc.iterrows():
        ax.scatter(row['params_b'], row['est_age_years'],
                   color=FAMILY_COLORS[row['family']], s=150, zorder=5)
        ax.annotate(row['label'], (row['params_b'], row['est_age_years']),
                    textcoords='offset points', xytext=(6, 2), fontsize=8)

    ax.set_xscale('log')
    ax.set_xlabel('Parameters (B)', fontsize=11)
    ax.set_ylabel('Estimated age equivalent (years)', fontsize=11)
    ax.set_title('Vocab: estimated developmental age by model size', fontsize=12, fontweight='bold')
    ax.set_ylim(2, 9)

    legend_elements = [Patch(facecolor=c, label=f) for f, c in FAMILY_COLORS.items()]
    ax.legend(handles=legend_elements, title='Family', fontsize=9)

    plt.tight_layout()
    plt.savefig(ROOT / 'results' / 'cross_architecture' / 'developmental_age_vocab.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    print(vocab_acc[['label','family','params_b','vocab_acc','est_age_years']]
          .to_string(index=False, float_format='{:.2f}'.format))

## 6. Interpretation

Fill this section after running the experiment. Key questions to answer:

**H2 (architecture > size on image tasks):**
- Does InternVL3.5-2B outperform TinyLLaVA-3.1B and SmolVLM2-500M on Matrix Reasoning and TROG despite similar or larger sizes?
- Do the two SigLIP-backbone models (SmolVLM2, TinyLLaVA) cluster together on image tasks?

**H4 (text tasks scale differently than image tasks):**
- Is the slope of accuracy vs log(params) steeper and more consistent for EGMA Math and Theory of Mind than for TROG and Matrix Reasoning?
- Do image tasks show a step change (jump between encoder tiers) rather than a smooth curve?

**Human plurality alignment:**
- Do larger models agree with the human plurality more often than smaller ones?
- Is there a task where small models *disagree* with human majority choices more than chance would predict (evidence of a systematic model-specific bias)?

**Developmental profile:**
- What age equivalent does the best-performing model achieve on Vocab?
- Is the InternVL3.5-8B developmental age on Vocab higher than its smaller sibling, or has the encoder already saturated the task?